# PCam CNN Training on Google Colab

This notebook trains a CNN using train/validation/test splits.
**Prerequisite:** Upload `train_new.zip`, `val_new.zip`, `test_new.zip` and the three CSV files to Google Drive.

## 1. Clone Repository

In [ ]:
!git clone https://github.com/goodguyjuro/pcam-cancer-detection.git
%cd pcam-cancer-detection
!pwd
!ls -la

## 2. Install Dependencies

In [ ]:
!pip install -q torch torchvision numpy matplotlib pandas scikit-learn

## 3. Mount Google Drive and Copy Split Files

Update `DRIVE_PATH` to match where you saved them.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/My Drive/pcam_splits'
DATA_DIR = '/content/pcam-cancer-detection/data'
!mkdir -p "$DATA_DIR"
!cp "$DRIVE_PATH/train_new.zip" "$DATA_DIR/"
!cp "$DRIVE_PATH/val_new.zip" "$DATA_DIR/"
!cp "$DRIVE_PATH/test_new.zip" "$DATA_DIR/"
!cp "$DRIVE_PATH/labels_train.csv" "$DATA_DIR/"
!cp "$DRIVE_PATH/labels_val.csv" "$DATA_DIR/"
!cp "$DRIVE_PATH/labels_test.csv" "$DATA_DIR/"
!ls -lh "$DATA_DIR"

## 4. Unzip the Split Files

In [ ]:
%cd /content/pcam-cancer-detection/data

!unzip -q train_new.zip -d train_new_tif
!unzip -q val_new.zip -d val_new_tif
!unzip -q test_new.zip -d test_new_tif

from pathlib import Path
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm
import shutil

def convert_tif_folder_to_png(src_dir, dst_dir):
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    tif_files = sorted(list(src_dir.glob("*.tif")) + list(src_dir.glob("*.tiff")))

    print(f"Found {len(tif_files)} TIFF files in {src_dir}")

    failed_files = []

    for p in tqdm(tif_files, desc=f"Converting {src_dir.name}"):
        out_path = dst_dir / f"{p.stem}.png"

        if out_path.exists():
            continue

        try:
            if p.stat().st_size == 0:
                failed_files.append((str(p), "empty file"))
                continue

            with Image.open(p) as img:
                img = img.convert("RGB")
                img.save(out_path, format="PNG", optimize=False)

        except Exception as e:
            failed_files.append((str(p), repr(e)))
            continue

    print(f"Finished {src_dir.name}")
    print(f"Failed files: {len(failed_files)}")

    if failed_files:
        log_path = dst_dir.parent / f"{src_dir.name}_failed_files.txt"
        with open(log_path, "w") as f:
            for file_path, error in failed_files:
                f.write(f"{file_path}\t{error}\n")

        print(f"Failed file list saved to: {log_path}")

    return failed_files

convert_tif_folder_to_png("train_new_tif", "train_new")
convert_tif_folder_to_png("val_new_tif", "val_new")
convert_tif_folder_to_png("test_new_tif", "test_new")

# Optional: remove TIFF folders to save Colab disk space
shutil.rmtree("train_new_tif")
shutil.rmtree("val_new_tif")
shutil.rmtree("test_new_tif")

!ls -lh
%cd /content/pcam-cancer-detection

In [ ]:
failed_train = convert_tif_folder_to_png("train_new_tif", "train_new")
failed_val = convert_tif_folder_to_png("val_new_tif", "val_new")
failed_test = convert_tif_folder_to_png("test_new_tif", "test_new")

In [ ]:
#Optional calculate normalization stats - already implemented in src/data.py, but can be run here on Colab if needed
from pathlib import Path
from PIL import Image
from torchvision import transforms
from tqdm.auto import tqdm
import torch

train_dir = Path("/content/pcam-cancer-detection/data/train_new")
image_paths = sorted(train_dir.glob("*.png"))

to_tensor = transforms.ToTensor()

channel_sum = torch.zeros(3)
channel_sum_sq = torch.zeros(3)
num_pixels = 0

for p in tqdm(image_paths, desc="Computing mean/std from full train set"):
    with Image.open(p) as img:
        img = img.convert("RGB")
        x = to_tensor(img)  # [3, H, W], values 0–1

    channel_sum += x.sum(dim=(1, 2))
    channel_sum_sq += (x ** 2).sum(dim=(1, 2))
    num_pixels += x.shape[1] * x.shape[2]

mean = channel_sum / num_pixels
std = (channel_sum_sq / num_pixels - mean ** 2).sqrt()

print("Images used:", len(image_paths))
print("mean =", mean.tolist())
print("std  =", std.tolist())

## 5. Verify Split Data Structure

In [ ]:
from pathlib import Path

data_dir = Path("/content/pcam-cancer-detection/data")

print("Train PNG:", len(list((data_dir / "train_new").glob("*.png"))))
print("Val PNG:", len(list((data_dir / "val_new").glob("*.png"))))
print("Test PNG:", len(list((data_dir / "test_new").glob("*.png"))))

print("Failed train:", len(failed_train))
print("Failed val:", len(failed_val))
print("Failed test:", len(failed_test))

## 6. Train Baseline Model (v1)

In [ ]:
%cd /content/pcam-cancer-detection
!PYTHONPATH=/content/pcam-cancer-detection python scripts/train.py \
  --model_version 1 \
  --epochs 5 \
  --batch_size 128 \
  --learning_rate 0.0003 \
  --num_workers 2 \
  --amp \
  --data_dir data

## 7. Train Augmentation Model (v2)

In [ ]:
%cd /content/pcam-cancer-detection
!PYTHONPATH=/content/pcam-cancer-detection python scripts/train.py \
  --model_version 2 \
  --epochs 5 \
  --batch_size 128 \
  --learning_rate 0.0003 \
  --num_workers 2 \
  --amp \
  --data_dir data

## 8. Train Dropout Model (v3)

In [ ]:
%cd /content/pcam-cancer-detection
!PYTHONPATH=/content/pcam-cancer-detection python scripts/train.py \
  --model_version 3 \
  --epochs 5 \
  --batch_size 128 \
  --learning_rate 0.0003 \
  --num_workers 2 \
  --amp \
  --data_dir data

In [ ]:
# V4 model  with lower learning rate and more epochs
%cd /content/pcam-cancer-detection
!PYTHONPATH=/content/pcam-cancer-detection python scripts/train.py \
  --model_version 4 \
  --epochs 10 \
  --batch_size 128 \
  --learning_rate 0.0001 \
  --num_workers 2 \
  --amp \
  --data_dir data

In [ ]:
#best performing model (v1) with more epochs and lower learning rate
%cd /content/pcam-cancer-detection
!PYTHONPATH=/content/pcam-cancer-detection python scripts/train.py \
  --model_version 1 \
  --epochs 10 \
  --batch_size 128 \
  --learning_rate 0.0001 \
  --num_workers 2 \
  --amp \
  --data_dir data

## 9. View Results

In [ ]:
!ls -lh results/
!find results -type f -name '*.png'

In [ ]:
#best checkpoint on test set
%%writefile scripts/evaluate_checkpoint.py
import argparse
import sys
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

ROOT_DIR = Path(__file__).resolve().parents[1]
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.data import PCamDataset, get_transforms
from src.model import get_model


def parse_args():
    parser = argparse.ArgumentParser(description="Evaluate PCam checkpoint")
    parser.add_argument("--checkpoint", type=str, required=True)
    parser.add_argument("--model_version", type=int, default=1, choices=[1, 2, 3, 4])
    parser.add_argument("--image_dir", type=str, required=True)
    parser.add_argument("--labels_csv", type=str, required=True)
    parser.add_argument("--batch_size", type=int, default=128)
    parser.add_argument("--num_workers", type=int, default=2)
    parser.add_argument("--amp", action="store_true")
    return parser.parse_args()


def main():
    args = parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = args.amp and device.type == "cuda"

    print(f"Using device: {device}")
    print(f"AMP: {'ON' if use_amp else 'OFF'}")

    labels_df = pd.read_csv(args.labels_csv, dtype={"id": str})

    dataset = PCamDataset(
        image_dir=args.image_dir,
        labels_df=labels_df,
        transform=get_transforms(augment=False),
    )

    loader = DataLoader(
        dataset,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=args.num_workers > 0,
    )

    model = get_model(args.model_version).to(device)

    checkpoint = torch.load(args.checkpoint, map_location=device)

    if isinstance(checkpoint, dict):
        if "model_state_dict" in checkpoint:
            model.load_state_dict(checkpoint["model_state_dict"])
        elif "model_state" in checkpoint:
            model.load_state_dict(checkpoint["model_state"])
        elif "state_dict" in checkpoint:
            model.load_state_dict(checkpoint["state_dict"])
        else:
            model.load_state_dict(checkpoint)
    else:
        model.load_state_dict(checkpoint)

    model.eval()
    criterion = nn.CrossEntropyLoss()

    running_loss = 0.0
    correct = 0
    total = 0
    pred_counts = torch.zeros(2, dtype=torch.long)
    true_counts = torch.zeros(2, dtype=torch.long)
    confusion = torch.zeros(2, 2, dtype=torch.long)

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=use_amp):
                outputs = model(images)
                loss = criterion(outputs, labels)

            preds = outputs.argmax(dim=1)

            running_loss += loss.item() * images.size(0)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            pred_counts += torch.bincount(preds.cpu(), minlength=2)
            true_counts += torch.bincount(labels.cpu(), minlength=2)

            for t, p in zip(labels.cpu(), preds.cpu()):
                confusion[t, p] += 1

    test_loss = running_loss / total
    test_acc = correct / total

    print(f"Test loss: {test_loss:.4f}")
    print(f"Test accuracy: {test_acc:.4f}")
    print(f"True class counts: {true_counts.tolist()}")
    print(f"Predicted class counts: {pred_counts.tolist()}")
    print("Confusion matrix:")
    print("Rows = true labels, columns = predicted labels")
    print(confusion.tolist())


if __name__ == "__main__":
    main()

In [ ]:
# Evaluate best checkpoint on test set
!PYTHONPATH=/content/pcam-cancer-detection python scripts/evaluate_checkpoint.py \
  --checkpoint results/version_1/best_checkpoint.pth \
  --model_version 1 \
  --image_dir data/test_new \
  --labels_csv data/labels_test.csv \
  --batch_size 128 \
  --num_workers 2 \
  --amp